## __Aprendizaje no supervisado__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

__Asunto__: Gaussian Mixture Model (GMM)

***

In [ ]:
## Instalación de libreria
#!python -m pip install ucimlrepo

In [ ]:
## Librerias
from ucimlrepo import fetch_ucirepo 
from tqdm import tqdm
from itertools import product
import matplotlib.pyplot as plt
import seaborn as sns
from pandas import DataFrame

from sklearn.preprocessing import StandardScaler

from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

## Carga de datos

__Dataset:__

Estos datos son referenciados en [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/109/wine) que son los resultados de un análisis químico de vinos cultivados en una misma región de Italia pero derivados de cultivos diferentes. 

<center>
    <img src='https://media.licdn.com/dms/image/v2/D5612AQHwzFJW75X22A/article-cover_image-shrink_720_1280/article-cover_image-shrink_720_1280/0/1674384291032?e=2147483647&v=beta&t=IBguDw6af4l1PyaTlV6Rw1YilLGFbUdn2_y178PYdww' width=800>
</center>

In [ ]:
## Cargar objeto de datos
dataset = fetch_ucirepo(id=109)

## Extraer los features del dataset
data = dataset.data.features 

## Mostrar los primeros 5 registros
display(data.head())

## Escalado de los datos
## Instancia del modelo de escalado
scaler = StandardScaler()

## Ajuste del modelo y transformación de los datos
data_scaled = scaler.fit_transform(data)
print('(shape) data: {}'.format(data_scaled.shape))

## Clase GaussianMixture

```{python}
    GaussianMixture(n_components=1, 
                    covariance_type='full', 
                    tol=0.001, 
                    max_iter=100, 
                    n_init=1, 
                    random_state=None, 
                    verbose=0, 
                    verbose_interval=10)
```

| Parámetros | Descripción |
|------------|-------------|
| n_components | número de componentes (por defecto, 1). |
| covariance_type | tipo de matriz de covarianza: 'full' si cada componente tiene su propia matriz de covarianza, 'tied' si todos los componentes comparten una misma matriz de covarianza, 'diag' cada componente tiene su matriz de covarianza diagonal, 'spherical' si cada componente tiene su propia varianza (por defecto, 'full'). | 
| tol | umbral de convergencia para el algoritmo EM (por defecto, 0.001). |
| max_iter | número máximo de iteraciones para el algoritmo EM (por defecto, 100) |
| n_init | número de inicializaciones a ejecutar (por defecto, 1) |
| random_state | (Optional): semilla de aleatoriedad |
| verbose | 0 no mostrar salida de iteraciones, >1 mostrar detalles (por defecto, 0). |
| verbose_interval | mostrar salida cada ciertas iteraciones (por defecto, 10)|

<br>

| Atributos | Descripción |
|----------|-------------|
| weights_ | Retorna los pesos de cada componente. |
| means_ | Retorna el promedio de cada componente por feature (n_components, n_features). |
| covariances_ | Retorna la matriz de covarianza de cada componente. |
| converged_ | True si el modelo alcanzó su convergencia, False si no. | 
| n_iter_ | número de iteraciones usada en el ajuste del modelo. | 

<br>

|Funciones | Descripción |
|----------|-------------|
| fit(X) | Entrena el modelo con los parametros asignados.|
| predict(X) | Predice el cluster mas cercano a la que pertenece cada muestra. |
| aic(X) | calcula y retorna el criterio de información de Akaike con modelo ajustado en una data $X$. |
| bic(X) | calcula y retorna el criterio de información de Bayes con modelo ajustado en una data $X$. |



Referencia ilustrativa de los tipos de matriz de covarianza

<center>
    <img src=https://scikit-learn.org/stable/_images/sphx_glr_plot_gmm_covariances_001.png width=600>
</center>

In [ ]:
## Instancia del modelo
model = GaussianMixture(n_components=2, 
                        covariance_type="full", 
                        n_init=100,
                        random_state=9001)

## Ajuste del modelo
model.fit(data_scaled)

In [ ]:
## Mostrar los pesos (ponderaciones) de cada componente.
print(model.weights_)

In [ ]:
## Mostrar el promedio de cada componente por feature
DataFrame(model.means_)

In [ ]:
## Mostrar las matrices de convarianza
print('shape: {}\n'.format(model.covariances_.shape))
print(model.covariances_)

In [ ]:
## Asignación de cada observación a cluster.
prediccion = model.predict(data_scaled)
prediccion

In [ ]:
model.predict_proba(data_scaled)

#### Graficación de los puntos con etiquetas de cluster mediante PCA.

In [ ]:
## Coputo de PCA
pca_model = PCA(n_components=2).fit(data_scaled)
X_pca = pca_model.transform(data_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], 
                y=X_pca[:, 1],
                hue=prediccion)
plt.title('Visualización mediante PCA')
plt.xlabel('Componente 1')
plt.ylabel('Componente 2')
plt.tight_layout()
plt.show()

#### Buscando el número de componentes

In [ ]:
## En GMM se puede analizar meddiante el  AIC o BIC 

## Lista de valores para los hiperparámetros a analizar
lista_componente = range(1, 21)
lista_cov = ['full', 'tied', 'diag', 'spherical']

## Almacenador de resultados
output = {'n_componentes': [],
          'cov': [],
          'AIC': [],
          'BIC': []}

for k, cov in tqdm(list(product(lista_componente, lista_cov))):
    
    ## Instancia del modelo
    model = GaussianMixture(n_components=k, 
                            covariance_type=cov,
                            n_init=100,  
                            random_state=20231007)

    ## Ajuste del modelo
    model.fit(data_scaled)

    output['n_componentes'].append(k)
    output['cov'].append(cov)
    output['AIC'].append(model.aic(data_scaled))
    output['BIC'].append(model.bic(data_scaled))

output = DataFrame(output)
output

In [ ]:
## Mostrar ambas medidas en graficas

plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
sns.lineplot(data=output, x='n_componentes', y='AIC', hue='cov')

plt.subplot(1, 2, 2)
sns.lineplot(data=output, x='n_componentes', y='BIC', hue='cov')

plt.tight_layout()
plt.show()

In [ ]:
print('La menor AIC es alcanzado con:')
print(output.sort_values(by='AIC', ascending=True).iloc[0])

print('\nLa menor BIC es alcanzado con:')
print(output.sort_values(by='BIC', ascending=True).iloc[0])

## Mejor modelo

In [ ]:
## Instancia del modelo
model = GaussianMixture(n_components=4, 
                        covariance_type="diag", 
                        random_state=9001)

## Ajuste del modelo
model.fit(data_scaled)

## Asignación de cada observación a cluster.
prediccion = model.predict(data_scaled)
prediccion

#### Visualización en 2D mediante PCA

In [ ]:
## Coputo de PCA
pca_model = PCA(n_components=2).fit(data_scaled)
X_pca = pca_model.transform(data_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], 
                y=X_pca[:, 1],
                hue=prediccion,
                palette=sns.color_palette())
plt.title('Visualización mediante PCA')
plt.xlabel('Componente 1')
plt.ylabel('Componente 2')
plt.tight_layout()
plt.show()